# Scraping Ulasan Google Play Store

Notebook ini mengambil ulasan pengguna sebuah aplikasi di Google Play Store secara mandiri
menggunakan library `google-play-scraper`, lalu menyimpannya ke `dataset_ulasan.csv`.

- **Sumber data:** ulasan publik aplikasi di Google Play Store (bahasa Indonesia)
- **Yang disimpan:** isi ulasan, rating, tanggal, jumlah like, versi aplikasi
- **Yang tidak disimpan:** nama dan foto profil pengguna (demi privasi)

In [6]:
%pip install -q google-play-scraper

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [7]:
import time

import pandas as pd
from google_play_scraper import Sort, app, reviews

## 1. Konfigurasi

`APP_ID` bisa diganti sesuai pilihan. ID aplikasi ada di URL Play Store setelah `id=`,
misalnya `https://play.google.com/store/apps/details?id=com.gojek.app`.

In [8]:
APP_ID = "com.gojek.app"      # ganti dengan aplikasi pilihanmu
BAHASA = "id"
NEGARA = "id"
TARGET_ULASAN = 100_000       # makin banyak data, makin tinggi akurasi model (minimal 10.000 setelah dibersihkan)
UKURAN_BATCH = 200            # jumlah ulasan per permintaan (maksimum 200)
JEDA_DETIK = 0.3              # jeda antarpermintaan agar tidak membebani server
OUTPUT_CSV = "dataset_ulasan.csv"

info = app(APP_ID, lang=BAHASA, country=NEGARA)
print("Aplikasi      :", info["title"])
print("Total ulasan  :", info["reviews"])
print("Rating        :", round(info["score"], 2))

Aplikasi      : Gojek - Transportasi & Makanan
Total ulasan  : 1899188
Rating        : 4.67


## 2. Proses scraping

Ulasan diambil per batch dari yang terbaru. Setiap batch mengembalikan *continuation token*
yang dipakai untuk mengambil halaman berikutnya, sampai target tercapai atau ulasan habis.

In [9]:
def scrape_ulasan(app_id, target, bahasa="id", negara="id", ukuran_batch=200, jeda=0.5, maks_gagal=5):
    """Mengambil ulasan Play Store secara bertahap sampai `target` tercapai."""
    hasil, token, gagal = [], None, 0

    while len(hasil) < target:
        try:
            batch, token = reviews(
                app_id,
                lang=bahasa,
                country=negara,
                sort=Sort.NEWEST,
                count=ukuran_batch,
                continuation_token=token,
            )
        except Exception as e:
            gagal += 1
            print(f"Permintaan gagal ({gagal}/{maks_gagal}): {e}")
            if gagal >= maks_gagal:
                break
            time.sleep(5)
            continue

        if not batch:
            print("Tidak ada ulasan lagi.")
            break

        gagal = 0
        hasil.extend(batch)
        if len(hasil) % 5000 < ukuran_batch:
            print(f"Terkumpul {len(hasil):,} ulasan")

        if token is None or token.token is None:
            print("Sudah mencapai halaman terakhir.")
            break
        time.sleep(jeda)

    return hasil[:target]


mulai = time.time()
data_mentah = scrape_ulasan(APP_ID, TARGET_ULASAN, BAHASA, NEGARA, UKURAN_BATCH, JEDA_DETIK)
print(f"\nSelesai: {len(data_mentah):,} ulasan dalam {(time.time() - mulai) / 60:.1f} menit")

Terkumpul 5,000 ulasan
Terkumpul 10,000 ulasan
Terkumpul 15,000 ulasan
Terkumpul 20,000 ulasan
Terkumpul 25,000 ulasan
Terkumpul 30,000 ulasan
Terkumpul 35,000 ulasan
Terkumpul 40,000 ulasan
Terkumpul 45,000 ulasan
Terkumpul 50,000 ulasan
Terkumpul 55,000 ulasan
Terkumpul 60,000 ulasan
Terkumpul 65,000 ulasan
Terkumpul 70,000 ulasan
Terkumpul 75,000 ulasan
Terkumpul 80,000 ulasan
Terkumpul 85,000 ulasan
Terkumpul 90,000 ulasan
Terkumpul 95,000 ulasan
Terkumpul 100,000 ulasan

Selesai: 100,000 ulasan dalam 7.4 menit


## 3. Simpan ke CSV

Hanya kolom yang relevan untuk analisis sentimen yang disimpan.

In [10]:
KOLOM = ["reviewId", "content", "score", "thumbsUpCount", "appVersion", "at"]

df = pd.DataFrame(data_mentah)[KOLOM]
df = df.dropna(subset=["content"]).drop_duplicates(subset="reviewId").reset_index(drop=True)
df.to_csv(OUTPUT_CSV, index=False)

print("Jumlah baris :", len(df))
print("Rentang waktu:", df["at"].min(), "s.d.", df["at"].max())
print("Tersimpan ke :", OUTPUT_CSV)

Jumlah baris : 100000
Rentang waktu: 2024-10-23 08:06:31 s.d. 2026-10-03 07:16:08
Tersimpan ke : dataset_ulasan.csv


In [11]:
df.head()

,reviewId,content,score,thumbsUpCount,appVersion,at
0,7244bba7-bf14-4445-ad75-7361329f372b,tombol untuk ngirim foto di chat suka hilang s...,1,0,5.77.1,2026-10-03 07:16:08
1,027608bc-1667-4478-910b-18d871803839,"top pokok e,terbaik di Indonesia",5,0,5.77.1,2026-10-03 06:53:36
2,f3dbb469-af6d-41f5-b612-a0299852a831,Cepat tanggap dan nyaman,5,0,5.77.1,2026-10-03 05:37:12
3,b7e04717-3911-4020-b8d7-0199bf96b013,good,5,0,5.77.1,2026-10-03 05:28:17
4,e8b62ee0-83dd-4ec2-b6c9-bd8c4f8e90b5,tolong orderan di aplikasi ini harus pakai KTP...,1,0,None,2026-10-03 05:17:57


In [12]:
df["score"].value_counts().sort_index()

score
1    26110
2     4008
3     3750
4     4858
5    61274
Name: count, dtype: int64